# Chapter 34 bridge: self-attention against autograd

Chapter 34 derives the self-attention backward pass by hand -- the hardest derivation in the book, and the last one it does. Every one of its gradients is checked here against autograd on the identical computation, including the gradient with respect to the input, which is the one a deeper stack would need.

Nothing here is reimplemented. The chapter's own file is executed, its own weights and data are handed to PyTorch, and the chapter's hand-derived gradients are compared against autograd. Tolerances are stated per check and are **relative** to the size of the quantity being compared, because an absolute threshold means nothing without a scale.

Run order: top to bottom, from a fresh kernel. Requires `requirements-bridges.txt` on top of the book's own `requirements.txt`.

In [1]:
import os, sys, numpy as np, torch
torch.set_default_dtype(torch.float64)          # match NumPy's float64 exactly
CH = os.path.join("..", "code", "ch34")
os.chdir(CH) if os.path.basename(os.getcwd()) != "ch34" else None
def run(name):
    exec(open(name, encoding="utf-8").read(), globals())
run("_lib.py")
print("chapter:", os.path.basename(os.getcwd()), "| torch", torch.__version__, "| numpy", np.__version__)


chapter: ch34 | torch 2.14.0 | numpy 2.4.4


In [2]:
def report(name, ours, theirs, tol=1e-9):
    a = np.asarray(ours, dtype=float); b = np.asarray(theirs, dtype=float)
    denom = max(np.abs(b).max(), 1e-300)
    absd = np.abs(a - b).max(); rel = absd / denom
    ok = rel <= tol
    RESULTS.append(dict(check=name, max_abs=float(absd), max_rel=float(rel),
                        scale=float(denom), tol=tol, passed=bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name:52s} max|diff| {absd:.3e}   "
          f"relative {rel:.2e}   (tolerance {tol:g})")
    return ok
RESULTS = []
MEASUREMENTS = []      # reported, never asserted: these have no single right answer


In [3]:
run("c1.py"); run("c2.py"); run("c3.py"); run("c4.py"); run("c5.py")   # c5 defines attn_forward/attn_backward

X (input)   (5, 8)
Q, K, V     (5, 8)   one query, key, and value PER POSITION

Q @ K.T     (5, 5)   position i's query dotted with every key
scores[2]   [ 1.38  2.89  0.56 -1.56  1.78]   <- how position 2 scores all 5 positions

attention weights for position 2: [0.133 0.603 0.059 0.007 0.198]   sums to 1.0000
output for position 2 = weighted sum of all 5 value vectors: [-0.133 -0.005 -0.686 -0.067]...
   d_k  score variance, unscaled   score variance, scaled
     8                       8.3                    1.037
    32                      31.4                    0.981
   128                     127.1                    0.993
   512                     501.8                    0.980

unscaled variance grows linearly with d_k, exactly as an
unscaled weighted sum's variance grew with fan-in in Chapter 31.
dividing by sqrt(d_k) keeps it near 1 regardless of dimension.

five real key vectors scored against one query, d_k = 64:
raw scores:       [ 9.78 17.58 14.82 -6.37  4.69]
unscaled

### Forward pass

In [4]:
rng = np.random.default_rng(11)
T, D = 6, 8
X = rng.normal(0, 1, (T, D))
Wq = rng.normal(0, 0.3, (D, D)); Wk = rng.normal(0, 0.3, (D, D)); Wv = rng.normal(0, 0.3, (D, D))
out, cache = attn_forward(X, Wq, Wk, Wv)

tX = torch.tensor(X, requires_grad=True)
tWq = torch.tensor(Wq, requires_grad=True); tWk = torch.tensor(Wk, requires_grad=True)
tWv = torch.tensor(Wv, requires_grad=True)
Q = tX @ tWq; K = tX @ tWk; V = tX @ tWv
A = torch.softmax((Q @ K.T) / np.sqrt(D), dim=-1)
tout = A @ V
report("self-attention forward", out, tout.detach().numpy(), 1e-12)

PASS  self-attention forward                               max|diff| 1.665e-16   relative 2.73e-16   (tolerance 1e-12)


np.True_

### Every hand-derived gradient

In [5]:
dout = rng.normal(0, 1, out.shape)
dX, dWq, dWk, dWv = attn_backward(dout, cache, Wq, Wk, Wv)
tout.backward(torch.tensor(dout))
report("dWq  (query projection)", dWq, tWq.grad.numpy())
report("dWk  (key projection)",   dWk, tWk.grad.numpy())
report("dWv  (value projection)", dWv, tWv.grad.numpy())
report("dX   (gradient to the input, what a deeper stack needs)", dX, tX.grad.numpy())

PASS  dWq  (query projection)                              max|diff| 2.776e-16   relative 1.61e-16   (tolerance 1e-09)
PASS  dWk  (key projection)                                max|diff| 4.441e-16   relative 3.58e-16   (tolerance 1e-09)
PASS  dWv  (value projection)                              max|diff| 4.441e-16   relative 1.13e-16   (tolerance 1e-09)
PASS  dX   (gradient to the input, what a deeper stack needs) max|diff| 3.331e-16   relative 2.50e-16   (tolerance 1e-09)


np.True_

### The softmax Jacobian on its own
The step most often got wrong. The chapter's row-wise form is compared with autograd directly, away from the rest of the computation.

In [6]:
s = rng.normal(0, 1, (5, 7))
w = softmax(s)
g = rng.normal(0, 1, w.shape)
ours = w * (g - (g * w).sum(1, keepdims=True))
ts = torch.tensor(s, requires_grad=True)
torch.softmax(ts, dim=1).backward(torch.tensor(g))
report("softmax Jacobian, row-wise", ours, ts.grad.numpy())

PASS  softmax Jacobian, row-wise                           max|diff| 1.110e-16   relative 1.82e-16   (tolerance 1e-09)


np.True_

In [7]:
import json
n_pass = sum(1 for r in RESULTS if r["passed"])
print(f"\n{n_pass} of {len(RESULTS)} ASSERTED checks passed")
if MEASUREMENTS:
    print(f"{len(MEASUREMENTS)} reported measurement(s), not asserted:")
    for m in MEASUREMENTS: print("   ", m)
print(json.dumps(dict(checks=RESULTS, measurements=MEASUREMENTS), indent=1))
assert n_pass == len(RESULTS), "a gradient check failed"



6 of 6 ASSERTED checks passed
{
 "checks": [
  {
   "check": "self-attention forward",
   "max_abs": 1.6653345369377348e-16,
   "max_rel": 2.730201109167619e-16,
   "scale": 0.6099677167904529,
   "tol": 1e-12,
   "passed": true
  },
  {
   "check": "dWq  (query projection)",
   "max_abs": 2.7755575615628914e-16,
   "max_rel": 1.611083761025566e-16,
   "scale": 1.722789111719466,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "dWk  (key projection)",
   "max_abs": 4.440892098500626e-16,
   "max_rel": 3.582779925282007e-16,
   "scale": 1.2395101544371514,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "dWv  (value projection)",
   "max_abs": 4.440892098500626e-16,
   "max_rel": 1.1348136885822565e-16,
   "scale": 3.913322638933545,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "dX   (gradient to the input, what a deeper stack needs)",
   "max_abs": 3.3306690738754696e-16,
   "max_rel": 2.500027976443085e-16,
   "scale": 1.3322527208732198,
   "tol": 1e-09,
 